In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

### Initialize Vector Index and Add Embeddings

In [67]:
context_texts = [
    ""
]

prompt_template = PromptTemplate(
    input_variables=["question", "context"],
    template="Question: {question}\nContext: {context}\nAnswer:"
)

In [61]:
import faiss
import numpy as np
import re

doc_embeddings = embed_model.encode(context_texts)
doc_embeddings = doc_embeddings.astype("float32")

index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Indexed {index.ntotal} context vectors.")

Indexed 1 context vectors.


### Define Semantic Search Function

In [62]:
def semantic_search(query_embedding, index, top_k=5):
    query_embedding = query_embedding.reshape(1, -1).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return indices

### Query Embedding and Retrieval

In [63]:
#query_embedding = np.random.random((1, 768)).astype('float32')
#retrieved_indices = semantic_search(query_embedding, index)
#print(f"Retrieved document indices: {retrieved_indices}")

### Initialize Tokenizer and LLM Model

In [64]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

### Create Prompt with Retrieval Context

### Initialize Memory and Build Chat Function

In [65]:
memory = ConversationBufferMemory(
    memory_key="chat_history", return_messages=False)


import ollama
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def user_ask(question):
    query_embedding = embed_model.encode(question)

    # FIX: reshape to 2D
    query_embedding = query_embedding.reshape(1, -1).astype("float32")
    retrieved_indices = semantic_search(query_embedding, index)
    context_texts = [f"Document {i}" for i in retrieved_indices[0]]

    chat_history = memory.load_memory_variables({}).get("chat_history", "")

    prompt = prompt_template.format(
        chat_history=chat_history,
        question=question,
        context="\n".join(context_texts)
    )

    response = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    answer = response["message"]["content"]

    memory.chat_memory.add_user_message(question)
    memory.chat_memory.add_ai_message(answer)

    return answer

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Generate Response

In [68]:
print(user_ask('Hello'))

Hello. I'm here to help you with your question or task, but it seems like there might be some confusion in the information provided. It appears you're listing documents as if they are interactions or a sequence of events, but they don't have any direct relevance to a specific answer.

Could you please clarify what you're trying to ask or accomplish? I'll do my best to provide assistance based on the context you provide.
